# RG3 F1: 학습 클래스 비용/사전비율 점수 조정

102/103 RF/ET(class_weight=balanced) 저장 점수 재사용. 각ref에서 학습 양성/음성 개수 및 보조sample_weight로 유효 설계 prior odds를 계산. target odds는 해당ref RG3 양성/음성 비율만 사용. logit(p)+log(target odds)-log(design odds) 변환. 클래스 조건부 분포/확률 추정이 정확하다는 가정 아래 Bayes prior 조정이지만, 제품간 가스 조건부 분포 동일성을 보장하지 않으므로 실제 calibration된 확률이라고 주장하지 않는다.

raw/corrected와 전문가 단독/쌍평균 및 임계값을 내부OOF F1에서 선택. RG3 평가코호트 정상601/가스19/같은7개구간 고정. 모델·순위·유효prior 계산에서 외부 구간 제외. CN7는 보조학습에만 사용. 새 독립 검증 아닌 개발용 중첩검증.


In [1]:
from pathlib import Path
import numpy as np,pandas as pd
from scipy.special import expit,logit
from sklearn.metrics import confusion_matrix,precision_recall_curve,average_precision_score
root=Path.cwd()
if not (root/'data').exists():root=root.parent
t=root/'reports/tables'
a=np.load(t/'rg3_CN7_support_fold_cache.npz',allow_pickle=False)
b=np.load(t/'rg3_CN7_gas_support_fold_cache.npz',allow_pickle=False)
y=a['labels'];groups=a['groups'];times=a['timestamps'];blocks=np.unique(groups)
assert len(y)==620 and y.sum()==19 and np.array_equal(y,b['labels']) and np.array_equal(times,b['timestamps'])
cache={}
for key in a.files:
 if key.startswith('ref_'):
  n=key.split('_')[1];ref=a[key].astype(int)
  matching=[k for k in b.files if k.startswith('ref_') and np.array_equal(b[k],ref)]
  assert len(matching)==1
  other=b['scores_'+matching[0].split('_')[1]]
  raw=np.column_stack([a['scores_'+n],other[:,1:]])
  assert np.allclose(raw[:,0],other[:,0])
  pos=float(y[ref].sum());neg=float(len(ref)-pos);target_odds=pos/neg
  design=[1.]
  for w in [.25,.5,1.]:design.append(((pos+10*w)/(pos+10))/((neg+695*w)/(neg+695)))
  for w in [.25,.5,1.]:design.append((pos+10*w)/(pos+10))
  corrected=expit(logit(np.clip(raw,1e-5,1-1e-5))+np.log(target_odds)-np.log(np.array(design)))
  cache[tuple(ref)]={'raw':raw,'training_prior_corrected':corrected}
def get(ref,q,method):
 assert set(groups[ref]).isdisjoint(groups[q])
 return cache[tuple(ref)][method][q]
def stats(yy,p):
 tn,fp,fn,tp=confusion_matrix(yy,p,labels=[0,1]).ravel()
 return dict(TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0,recall=tp/(tp+fn) if tp+fn else 0)
def cutfit(yy,score):
 p,r,c=precision_recall_curve(yy,score);f=np.divide(2*p[:-1]*r[:-1],p[:-1]+r[:-1],out=np.zeros_like(c),where=p[:-1]+r[:-1]>0)
 cut=float(c[np.lexsort((c,r[:-1],f))[-1]]);return cut,stats(yy,score>=cut)
mixes=[np.eye(7)[i] for i in range(7)]+[(np.eye(7)[i]+np.eye(7)[j])/2 for i in range(7) for j in range(i+1,7)]
records=[];choices=[]
for block in blocks:
 q=np.flatnonzero(groups==block);ref=np.flatnonzero(groups!=block);candidates=[];localbest=[]
 for method in ['raw','training_prior_corrected']:
  out=get(ref,q,method);inner=np.full((len(ref),7),np.nan)
  for ib in np.unique(groups[ref]):
   mask=groups[ref]==ib;inner[mask]=get(ref[~mask],ref[mask],method)
  local=[]
  for w in mixes:
   cut,result=cutfit(y[ref],inner@w);item=(result['F1'],result['recall'],-result['FP'],method,w,cut,out@w)
   candidates.append(item);local.append(item)
  localbest.append(max(local,key=lambda v:v[:3]))
 best=max(candidates,key=lambda v:v[:3])
 named=[(v[3]+'_inner_F1_blend',v) for v in localbest]+[('inner_F1_selected_prior_and_blend',best)]
 for name,v in named:
  for j,i in enumerate(q):records.append(dict(model=name,TimeStamp=times[i],block=int(block),actual=int(y[i]),prediction=int(v[6][j]>=v[5]),score=float(v[6][j]),threshold=v[5]))
 choices.append(dict(block=int(block),method=best[3],weights=','.join(map(str,best[4])),threshold=best[5],inner_F1=best[0]))
p=pd.DataFrame(records);m=pd.DataFrame([dict(model=name,AP=average_precision_score(g.actual,g.score),**stats(g.actual,g.prediction)) for name,g in p.groupby('model')])
for obj,name in [(p,'predictions'),(m,'metrics'),(pd.DataFrame(choices),'inner_choices')]:obj.to_csv(t/f'rg3_prior_odds_{name}.csv',index=False,encoding='utf-8-sig')
print(m.to_string(index=False))


                                  model       AP  TP  FP  FN  TN       F1   recall
      inner_F1_selected_prior_and_blend 0.054722   4  34  15 567 0.140351 0.210526
                     raw_inner_F1_blend 0.070021   6  41  13 560 0.181818 0.315789
training_prior_corrected_inner_F1_blend 0.065005   1  28  18 573 0.041667 0.052632
